# Módulo 04 · Aula 4 — Subconsultas, CTEs e funções de janela

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

Até aqui, cada consulta foi um bloco só. Esta aula trata de **consultas em etapas** — e
depois apresenta a ferramenta mais poderosa do SQL moderno, a que a maioria dos cursos
introdutórios não chega a mencionar.

As **funções de janela** resolvem uma classe de problema que, sem elas, é dolorosa:
calcular retorno diário, média móvel, total acumulado, ranking dentro de cada grupo. Em
finanças, é metade do trabalho. Se você aprender só uma coisa deste módulo além do
`GROUP BY`, aprenda esta.

Ao final desta aula você vai:

- quebrar uma consulta grande em etapas legíveis com **CTEs** (`WITH`);
- usar subconsultas no `WHERE` e no `FROM`;
- entender a diferença entre `GROUP BY` e uma **janela**;
- calcular retorno diário com `LAG`, ranking com `ROW_NUMBER` e média móvel com
  `ROWS BETWEEN`;
- resolver o problema do "top N por grupo", que não tem solução limpa sem janelas.

**Tempo estimado:** 90 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê o banco de dados da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "04_SQL"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 140)

conexao = sqlite3.connect("../data/capacitacao.db")


def consultar(sql):
    return pd.read_sql_query(sql, conexao)


print("Conectado. SQLite", sqlite3.sqlite_version)

## 1. Subconsulta no `WHERE`

Uma **subconsulta** é uma consulta dentro de outra. A forma mais comum responde a
perguntas do tipo "acima da média" — que exigem calcular a média antes de poder comparar
com ela.

In [ ]:
consultar("""
    SELECT
        ticker,
        ROUND(AVG(fechamento_ajustado), 2) AS preco_medio
    FROM cotacoes
    GROUP BY ticker
    HAVING AVG(fechamento_ajustado) > (
        SELECT AVG(fechamento_ajustado) FROM cotacoes
    )
    ORDER BY preco_medio DESC
""")

A subconsulta entre parênteses roda primeiro, devolve **um número**, e esse
número entra na comparação. Isso é impossível de escrever em uma consulta simples: você
precisaria saber a média antes de escrever a consulta.

Subconsulta também funciona com `IN`, devolvendo uma **lista**:

In [ ]:
consultar("""
    SELECT ticker, data, ROUND(fechamento_ajustado, 2) AS fechamento
    FROM cotacoes
    WHERE ticker IN (
        SELECT ticker FROM empresas WHERE controle = 'Estatal'
    )
      AND data = '2025-12-30'
""")

## 2. CTE: dando nome às etapas

Subconsultas aninhadas ficam ilegíveis rápido. A **CTE** (*Common Table Expression*), com
a palavra `WITH`, resolve isso: ela dá **nome** a um resultado intermediário, que passa a
ser usado como se fosse uma tabela.

```sql
WITH nome_da_etapa AS (
    SELECT ...
)
SELECT ... FROM nome_da_etapa
```

Você já viu CTEs de passagem na aula 3. Agora com atenção — porque escrever SQL legível é
o que separa uma consulta que você entende daqui a seis meses de uma que você reescreve do
zero.

Lembra do problema da aula 1, em que o apelido do `SELECT` não podia ser usado no `WHERE`?
A CTE é a solução portátil.

In [ ]:
consultar("""
    WITH com_amplitude AS (
        SELECT
            ticker,
            data,
            maxima - minima AS amplitude
        FROM cotacoes
    )
    SELECT *
    FROM com_amplitude
    WHERE amplitude > 5           -- agora o nome existe de verdade
    ORDER BY amplitude DESC
    LIMIT 5
""")

O cálculo é escrito **uma vez**, ganha nome, e o filtro usa o nome. Funciona no
SQLite, no Postgres, em todo lugar.

### CTEs encadeadas: o pipeline

Várias CTEs, separadas por vírgula, formam uma sequência de etapas — e cada uma pode usar
as anteriores. É o equivalente SQL de encadear operações de pandas, e transforma uma
consulta monolítica em algo que se lê de cima para baixo.

In [ ]:
consultar("""
    WITH
    -- etapa 1: reduzir a granularidade para mensal
    mensal AS (
        SELECT
            ticker,
            strftime('%Y-%m', data)            AS ano_mes,
            AVG(fechamento_ajustado)           AS preco_medio
        FROM cotacoes
        GROUP BY ticker, ano_mes
    ),
    -- etapa 2: acrescentar o setor
    com_setor AS (
        SELECT
            m.*,
            e.setor
        FROM mensal m
        INNER JOIN empresas e ON m.ticker = e.ticker
    ),
    -- etapa 3: resumir por setor e mês
    por_setor AS (
        SELECT
            setor,
            ano_mes,
            ROUND(AVG(preco_medio), 2) AS preco_medio_setor
        FROM com_setor
        GROUP BY setor, ano_mes
    )
    SELECT *
    FROM por_setor
    WHERE ano_mes >= '2025-10'
    ORDER BY ano_mes DESC, setor
""")

Leia de cima para baixo: agregar, enriquecer, resumir, filtrar. Cada etapa é
pequena e tem nome. Se o número final vier errado, você troca o `SELECT` final por
`SELECT * FROM mensal` e inspeciona a etapa 1 — depuração passo a passo, dentro da própria
consulta.

> **Comentários em SQL:** `--` comenta até o fim da linha, e `/* ... */` comenta um bloco.
> Em consulta de vinte linhas, comentar cada etapa é o que a torna legível para a próxima
> pessoa — inclusive você.

## 3. Funções de janela: a ideia

Aqui está a virada da aula.

`GROUP BY` **colapsa**: 1.246 pregões da PETR4 viram uma linha. Você ganha o resumo e perde
o detalhe.

Uma **função de janela** calcula um valor agregado **sem colapsar**: cada linha continua
existindo e ganha uma coluna nova, calculada olhando para um conjunto de linhas vizinhas —
a *janela*.

| | `GROUP BY` | Função de janela |
|---|---|---|
| Linhas no resultado | uma por grupo | **todas** |
| Serve para | resumir | comparar cada linha com seu contexto |
| Exemplo | preço médio por papel | quanto **esta** linha se afasta da média do papel |

A sintaxe é `função() OVER (...)`:

```sql
AVG(fechamento) OVER (PARTITION BY ticker ORDER BY data)
                      └── o grupo ──┘  └── a ordem ──┘
```

- **`PARTITION BY`** faz o papel do `GROUP BY`: define os baldes. Sem ele, a janela é a
  tabela inteira.
- **`ORDER BY`** define a ordem *dentro* da janela — essencial para tudo que envolve
  "anterior", "acumulado" ou "posição".

In [ ]:
# Cada pregão, ao lado da média do próprio papel — sem perder as linhas.
# A janela é calculada na CTE, sobre a série inteira; o recorte de data vem depois.
consultar("""
    WITH com_media AS (
        SELECT
            data,
            ticker,
            fechamento_ajustado,
            AVG(fechamento_ajustado) OVER (PARTITION BY ticker) AS media_do_papel
        FROM cotacoes
    )
    SELECT
        data,
        ticker,
        ROUND(fechamento_ajustado, 2)                    AS fechamento,
        ROUND(media_do_papel, 2)                         AS media_do_papel,
        ROUND(fechamento_ajustado - media_do_papel, 2)   AS desvio
    FROM com_media
    WHERE ticker = 'PETR4'
      AND data >= '2025-12-20'
    ORDER BY data
""")

A coluna `media_do_papel` repete o mesmo valor em toda linha — é a média dos
1.246 pregões da PETR4 —, mas **as linhas individuais continuam lá**. Com `GROUP BY`, isso
exigiria uma consulta separada e um `JOIN` de volta.

> **Repare na estrutura:** a janela foi calculada em uma CTE, sobre a tabela inteira, e o
> recorte de data veio no `SELECT` de fora. Isso não é capricho — a seção 6 mostra o que
> acontece quando se inverte a ordem.

> **Versão do SQLite.** Funções de janela existem a partir do SQLite 3.25 (2018). A célula
> acima já mostrou a versão instalada; qualquer Python moderno tem folga.

## 4. `LAG` e `LEAD`: olhar a linha vizinha

`LAG(coluna)` devolve o valor da **linha anterior**; `LEAD`, da seguinte. É a ferramenta
para variação de um período para o outro — o `.shift()` do pandas.

E é assim que se calcula retorno diário em SQL puro.

In [ ]:
consultar("""
    WITH com_anterior AS (
        SELECT
            data,
            ticker,
            fechamento_ajustado,
            LAG(fechamento_ajustado) OVER (
                PARTITION BY ticker      -- não misturar papéis diferentes
                ORDER BY data            -- "anterior" = pregão anterior
            ) AS fechamento_anterior
        FROM cotacoes
        WHERE ticker = 'PETR4'
    )
    SELECT
        data,
        ROUND(fechamento_anterior, 2) AS ontem,
        ROUND(fechamento_ajustado, 2) AS hoje,
        ROUND((fechamento_ajustado / fechamento_anterior - 1) * 100, 2) AS retorno_pct
    FROM com_anterior
    WHERE data >= '2025-12-20'
    ORDER BY data
""")

**O `PARTITION BY ticker` não é decoração.** Sem ele, a janela atravessaria a
fronteira entre papéis, e o primeiro pregão da VALE3 seria comparado com o último da PETR4
— produzindo um "retorno" absurdo, uma vez por papel. É o erro mais comum com `LAG`, e o
mesmo cuidado que o `groupby("ticker").pct_change()` do módulo 02 exigia.

Repare também que a primeira linha de cada papel tem `fechamento_anterior` vazio: não
existe pregão anterior. `NULL` aqui está **certo** — é a resposta honesta.

## 5. Ranking: `ROW_NUMBER`, `RANK`, `DENSE_RANK`

Três funções que numeram as linhas dentro de cada partição, diferindo apenas no tratamento
de empates.

| Função | Empate vira | Sequência com empate |
|---|---|---|
| `ROW_NUMBER()` | posições diferentes, arbitrárias | 1, 2, 3, 4 |
| `RANK()` | mesma posição, e **pula** as seguintes | 1, 2, 2, 4 |
| `DENSE_RANK()` | mesma posição, **sem pular** | 1, 2, 2, 3 |

In [ ]:
consultar("""
    SELECT
        e.setor,
        c.ticker,
        ROUND(c.fechamento_ajustado, 2) AS fechamento,
        ROW_NUMBER() OVER (PARTITION BY e.setor ORDER BY c.fechamento_ajustado DESC) AS posicao
    FROM cotacoes c
    INNER JOIN empresas e ON c.ticker = e.ticker
    WHERE c.data = '2025-12-30'
    ORDER BY e.setor, posicao
""")

### O problema do "top N por grupo"

Esta é a pergunta que motiva as janelas na prática:

> *Quais foram os 2 pregões de maior volume de cada papel?*

Com `GROUP BY` você consegue o **maior** de cada grupo (`MAX`), mas não os dois maiores.
Não há como. Com janela, é direto: numere dentro de cada partição, e depois filtre pela
numeração — em uma CTE, porque o `WHERE` não enxerga funções de janela (elas são
calculadas depois, junto do `SELECT`).

In [ ]:
consultar("""
    WITH ranqueado AS (
        SELECT
            ticker,
            data,
            volume,
            ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY volume DESC) AS posicao
        FROM cotacoes
    )
    SELECT ticker, data, volume, posicao
    FROM ranqueado
    WHERE posicao <= 2
    ORDER BY ticker, posicao
""")

16 linhas: 2 para cada um dos 8 papéis. Guarde este padrão —
**janela numa CTE, filtro fora** —, porque ele resolve uma família inteira de perguntas:
o cliente mais valioso de cada cidade, a maior venda de cada mês, os três piores dias de
cada ativo.

## 6. Acumulado e média móvel: o quadro da janela

Até agora a janela era a partição inteira. Dá para estreitá-la, dizendo quais linhas ao
redor da atual entram no cálculo:

```sql
ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
```

Isso significa "as 4 linhas anteriores mais esta" — cinco no total.

| Quadro | Significado |
|---|---|
| `UNBOUNDED PRECEDING AND CURRENT ROW` | do início até aqui — **acumulado** |
| `n PRECEDING AND CURRENT ROW` | janela móvel de n+1 linhas |
| `UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` | a partição toda (o padrão sem `ORDER BY`) |

> **Cuidado com o padrão.** Quando há `ORDER BY` na janela e você **não** escreve o quadro,
> o padrão do SQL é do início até a linha atual — ou seja, acumulado, não a partição
> inteira. É por isso que `AVG(x) OVER (PARTITION BY t ORDER BY d)` dá uma média que muda a
> cada linha, e não a média do grupo. Fonte inesgotável de confusão: se você quer a média
> do grupo inteiro, **não** ponha `ORDER BY` na janela.

### Primeiro, uma armadilha

Lembre da ordem de execução: `WHERE` roda **antes** do `SELECT`, e funções de janela são
calculadas junto do `SELECT`. Logo, **a janela só enxerga as linhas que sobreviveram ao
`WHERE`.**

Veja o que acontece se filtrarmos direto:

In [ ]:
consultar("""
    SELECT
        data,
        ROUND(fechamento_ajustado, 2) AS fechamento,
        ROUND(AVG(fechamento_ajustado) OVER (
            PARTITION BY ticker ORDER BY data
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ), 2) AS media_movel_5d,
        ROUND(MAX(fechamento_ajustado) OVER (
            PARTITION BY ticker ORDER BY data
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2) AS maxima_ate_hoje
    FROM cotacoes
    WHERE ticker = 'WEGE3'
      AND data >= '2025-12-15'      -- este filtro AMPUTA a janela
    ORDER BY data
""")

Duas coisas estão erradas aí, e nenhuma se denuncia:

- na primeira linha, `media_movel_5d` é **igual** ao próprio fechamento — a janela de 5
  dias não tinha 5 dias, tinha 1;
- `maxima_ate_hoje` começa em 48,23, mas a WEGE3 já valeu muito mais que isso no período.
  O "máximo histórico" só olhou para dezembro de 2025.

**A correção é calcular a janela sobre a série inteira e filtrar depois** — ou seja, janela
na CTE, `WHERE` do lado de fora. O mesmo padrão do "top N por grupo".

In [ ]:
consultar("""
    WITH com_janelas AS (
        SELECT
            data,
            ticker,
            fechamento_ajustado,
            AVG(fechamento_ajustado) OVER (
                PARTITION BY ticker ORDER BY data
                ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
            ) AS media_movel_5d,
            MAX(fechamento_ajustado) OVER (
                PARTITION BY ticker ORDER BY data
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS maxima_ate_hoje
        FROM cotacoes
        WHERE ticker = 'WEGE3'        -- este filtro é seguro: não corta a série no tempo
    )
    SELECT
        data,
        ROUND(fechamento_ajustado, 2) AS fechamento,
        ROUND(media_movel_5d, 2)      AS media_movel_5d,
        ROUND(maxima_ate_hoje, 2)     AS maxima_ate_hoje
    FROM com_janelas
    WHERE data >= '2025-12-15'
    ORDER BY data
""")

Agora a média móvel usa os cinco pregões de verdade, e `maxima_ate_hoje` mostra o
topo real da série.

> **A distinção que importa:** filtrar por `ticker` dentro da CTE é seguro, porque a janela
> é particionada por `ticker` mesmo — nada que interessa foi cortado. Filtrar por **data**
> é que amputa, porque a janela olha para trás no tempo. Antes de pôr um `WHERE` junto de
> uma janela, pergunte: *isso remove linhas de que a janela precisa?*

A coluna `maxima_ate_hoje` é o **máximo histórico acumulado** — em finanças, o
insumo do cálculo de *drawdown*, que mede quanto um ativo caiu desde o topo. Vamos até o
fim:

In [ ]:
consultar("""
    WITH com_topo AS (
        SELECT
            data,
            ticker,
            fechamento_ajustado,
            MAX(fechamento_ajustado) OVER (
                PARTITION BY ticker
                ORDER BY data
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS topo_ate_aqui
        FROM cotacoes
    ),
    com_queda AS (
        SELECT
            ticker,
            data,
            (fechamento_ajustado / topo_ate_aqui - 1) * 100 AS queda_pct
        FROM com_topo
    )
    SELECT
        ticker,
        ROUND(MIN(queda_pct), 1) AS pior_queda_desde_o_topo_pct
    FROM com_queda
    GROUP BY ticker
    ORDER BY pior_queda_desde_o_topo_pct
""")

Esse é o *maximum drawdown* de cada papel no período — uma medida de risco que
todo relatório de fundo traz, calculada inteiramente dentro do banco, em uma consulta.

MGLU3 aparece com a maior queda, o que é coerente com tudo que os módulos 02 e 03
mostraram sobre esse papel.

## 7. Conferindo contra o pandas

O teste de sempre, agora sobre o retorno diário com `LAG` × `pct_change`.

In [ ]:
via_sql = consultar("""
    WITH com_anterior AS (
        SELECT
            ticker,
            data,
            fechamento_ajustado,
            LAG(fechamento_ajustado) OVER (PARTITION BY ticker ORDER BY data) AS anterior
        FROM cotacoes
    )
    SELECT
        ticker,
        ROUND(AVG((fechamento_ajustado / anterior - 1) * 100), 4) AS retorno_medio_pct,
        COUNT(*) AS observacoes
    FROM com_anterior
    WHERE anterior IS NOT NULL
    GROUP BY ticker
    ORDER BY ticker
""")
via_sql

In [ ]:
acoes = (pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
           .sort_values(["ticker", "data"]))
acoes["retorno"] = acoes.groupby("ticker")["fechamento_ajustado"].pct_change() * 100

via_pandas = (
    acoes.groupby("ticker")["retorno"]
    .agg(retorno_medio_pct="mean", observacoes="count")
    .round({"retorno_medio_pct": 4})
    .reset_index()
)
via_pandas

In [ ]:
import numpy as np

print("Retornos batem  :", np.allclose(via_sql["retorno_medio_pct"], via_pandas["retorno_medio_pct"]))
print("Observações batem:", np.array_equal(via_sql["observacoes"].values, via_pandas["observacoes"].values))
print()
print("1.245 observações por papel = 1.246 pregões menos o primeiro, que não tem anterior.")

## 8. O mapa, completo

| Pergunta | pandas | SQL |
|---|---|---|
| etapas nomeadas | variáveis intermediárias | `WITH ... AS (...)` |
| valor da linha anterior | `.shift()` | `LAG() OVER (...)` |
| variação percentual | `.pct_change()` | `LAG` + divisão |
| média móvel | `.rolling(n).mean()` | `AVG() OVER (... ROWS BETWEEN n PRECEDING ...)` |
| acumulado | `.cumsum()`, `.cummax()` | `SUM()/MAX() OVER (... UNBOUNDED PRECEDING ...)` |
| ranking dentro do grupo | `.groupby().rank()` | `RANK() OVER (PARTITION BY ...)` |
| n maiores por grupo | `.groupby().head(n)` | `ROW_NUMBER()` numa CTE + `WHERE` |
| agregado sem colapsar | `.transform()` | `AVG() OVER (PARTITION BY ...)` |

A última linha merece atenção: `OVER (PARTITION BY ...)` é o `transform` do pandas. Se você
entendeu um, entendeu o outro.

## 9. Recapitulando

- **Subconsulta** é uma consulta dentro de outra; resolve "acima da média" e afins.
- **CTE (`WITH`)** dá nome a etapas intermediárias. Consultas encadeadas se leem de cima
  para baixo e se depuram etapa por etapa. Use sempre que passar de umas dez linhas.
- **`GROUP BY` colapsa; janela não.** Uma função de janela acrescenta coluna sem tirar
  linha.
- `PARTITION BY` é o balde; `ORDER BY` é a ordem dentro dele. **Esquecer o `PARTITION BY`
  em `LAG` mistura papéis diferentes** e produz números absurdos.
- `LAG`/`LEAD` olham a linha vizinha — é assim que se calcula retorno em SQL.
- `ROW_NUMBER` numa CTE, filtrada por fora, resolve o "top N por grupo", que não tem
  solução com `GROUP BY`.
- `ROWS BETWEEN` estreita a janela: média móvel, acumulado, drawdown.
- **Com `ORDER BY` na janela e sem quadro explícito, o padrão é acumulado, não o grupo
  inteiro.** Se você quer a média do grupo, não ponha `ORDER BY`.
- Funções de janela **não podem ser usadas no `WHERE`** — sempre CTE no meio.

**Próxima aula:** juntar as duas ferramentas — onde termina o SQL, onde começa o pandas, e
como escrever consultas com parâmetros sem abrir um buraco de segurança.

In [ ]:
conexao.close()
print("Conexão fechada.")